# RAMBHA-LP — Train the PINN on Google Colab (GPU)

Same job as the Kaggle notebook, adapted to Colab + Google Drive.

**Before running:** `Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU`.

**Get the code onto Colab** in either way (GUIDE.md §8):
* **Drive:** upload the `solution/` folder to your Google Drive, then mount it (cell below), or
* **Git:** `!git clone <your repo>` if you push the project to GitHub.

You only need `src/`, `config.yaml`, and (optionally) `outputs/cache/synth_*.npz`. The raw
RAMBHA archive is **not** needed for training.

In [ ]:
# --- Mount Google Drive (skip if you used git clone instead) ------------------------
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Point SOL at your solution folder ---------------------------------------------
import sys, os
# EDIT this to wherever you put the folder in Drive:
SOL = '/content/drive/MyDrive/RAMBHA/solution'
assert os.path.exists(os.path.join(SOL, 'config.yaml')), f'config.yaml not found under {SOL}'
sys.path.insert(0, os.path.join(SOL, 'src'))
print('solution dir:', SOL)

In [ ]:
# torch is preinstalled on Colab; pyyaml too. Install nothing unless a import fails.
import torch, numpy as np
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
from rambhalp.config import load_config
from rambhalp import synthetic, train
cfg = load_config(os.path.join(SOL, 'config.yaml'))
cfg['train']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg['train']['epochs'] = 200
cfg['train']['batch_size'] = 2048

cache = os.path.join(SOL, 'outputs', 'cache')
if os.path.exists(os.path.join(cache, 'synth_train.npz')):
    tr = np.load(os.path.join(cache, 'synth_train.npz'));  va = np.load(os.path.join(cache, 'synth_val.npz'))
    Xtr, Ytr, Xva, Yva = tr['X'], tr['Y'], va['X'], va['Y']
else:
    Xtr, Ytr, _ = synthetic.make_dataset(cfg['synthetic']['n_train'], cfg, seed=1337)
    Xva, Yva, _ = synthetic.make_dataset(cfg['synthetic']['n_val'], cfg, seed=1338)
print('train', Xtr.shape, 'val', Xva.shape)

In [ ]:
model, hist = train.train_model(Xtr, Ytr, cfg, X_val=Xva, Y_val=Yva, verbose=True)

In [ ]:
# --- Save straight back into the Drive solution folder so your laptop can sync it ---
import json, pathlib
ckpt_dir = pathlib.Path(SOL) / 'outputs' / 'checkpoints'; ckpt_dir.mkdir(parents=True, exist_ok=True)
tbl_dir = pathlib.Path(SOL) / 'outputs' / 'tables'; tbl_dir.mkdir(parents=True, exist_ok=True)
train.save_checkpoint(model, cfg, ckpt_dir / 'pinn.pt', extra={'kind': 'pinn', 'history': hist})
json.dump(hist, open(tbl_dir / 'train_history.json', 'w'), indent=2)
print('saved pinn.pt into', ckpt_dir)

Then locally run steps 05→07 (apply, validate, figures) against the downloaded checkpoint.